# 02 — Comprendre les sources avant de préparer la base

**Question du projet :** les générations récentes introduisent-elles des Pokémon aux statistiques plus élevées ou plus spécialisées ? Avant de comparer les générations, nous devons comprendre les observations et les informations disponibles.

**Choix retenu : une ligne par forme PokéAPI.** Nous conservons toutes les formes de l’instantané, même cosmétiques, Gigamax ou non définies par défaut. Garder le détail permettra ensuite des analyses par forme, variété, espèce ou génération. Les talents et les conditions d’évolution sont hors périmètre.

Les fonctions sont dans `src/exploration.py`. Les cellules suivent l’ordre **observation → explication → conclusion ou proposition de traitement**. Aucun fichier brut n’est modifié, aucun regroupement de formes n’est appliqué et aucune base finale n’est exportée. Les résultats chiffrés commentés concernent `reference_01`.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == "notebook":
    project_dir = project_dir.parent
assert (project_dir / "src" / "exploration.py").exists(), "Ouvrir depuis le projet ou notebook/."
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from src.collect import collect_sources, load_sources, check_sources
from src import exploration as eda

collect_sources()  # Vérifie et réutilise l’instantané partagé.
data = load_sources()
source_checks = check_sources(data)

Conservé et vérifié : pokemon_species.csv
Conservé et vérifié : pokemon.csv
Conservé et vérifié : pokemon_stats.csv
Conservé et vérifié : stats.csv
Conservé et vérifié : pokemon_species_names.csv
Conservé et vérifié : pokemon_types.csv
Conservé et vérifié : type_names.csv
Conservé et vérifié : showdown_pokedex.json
Conservé et vérifié : pokemon_forms.csv
Conservé et vérifié : pokemon_form_types.csv
Conservé et vérifié : version_groups.csv
Contrôles terminés : 1579 formes conservées ; aucun filtre ni regroupement.


## 1. Que représente une ligne ?

Une **espèce** peut avoir plusieurs **variétés**, et une variété plusieurs **formes**. Les six statistiques sont stockées au niveau de la variété. Nous rattachons ces informations aux formes par des jointures qui contrôlent le nombre de correspondances. La fonction vérifie que tous les `form_id` restent présents et uniques.

Les exemples ci-dessous sont choisis pour expliquer la structure ; le traitement lui-même s’applique à toutes les formes.

In [2]:
forms = eda.explore_forms(data)
display(eda.summarize_forms(forms))
species = eda.summarize_species(forms)
display(species.loc[species["species_id"].isin([493, 669, 745, 773, 925, 1007, 1008])])

,nombre
Formes,1579
Identifiants de forme distincts,1579
Variétés représentées,1351
Espèces représentées,1025
Formes non définies par défaut,236
Formes Gigamax,34
Variétés sans forme par défaut,8
Formes avec une statistique manquante,0
Formes sans génération renseignée,0
Formes datées avant leur espèce,0


,species_id,species_name,forms,varieties
492,493,arceus,19,1
668,669,flabebe,5,1
744,745,lycanroc,3,3
772,773,silvally,18,1
924,925,maushold,2,2
1006,1007,koraidon,5,5
1007,1008,miraidon,5,5


**Constat :** 1 579 formes représentent 1 351 variétés et 1 025 espèces. Flabébé a cinq formes pour une variété ; Lougaroc a trois formes pour trois variétés. Arceus a 19 formes et Silvallié 18, chacun pour une seule variété.

**Conclusion :** les identifiants `form_id`, `pokemon_id` et `species_id` désignent des objets différents. Nous gardons les 236 formes non définies par défaut et les 34 Gigamax. Les huit variétés de déplacement de Koraidon/Miraidon n’ont pas de forme marquée par défaut, mais leurs formes sont bien présentes : aucun traitement de remplacement n’est nécessaire pour les conserver.

## 2. Statistiques identiques et dates différentes : des doublons ?

Les six statistiques viennent de la variété. La génération vient du groupe de jeux d’introduction de **chaque forme**. Nous gardons la génération de l’espèce séparément pour ne pas confondre les dates.

In [3]:
examples = ["charizard", "charizard-mega-x", "charizard-gmax", "lycanroc-midday",
            "lycanroc-midnight", "lycanroc-dusk", "maushold-family-of-four", "maushold-family-of-three"]
display(forms.loc[forms["form_name"].isin(examples),
                  ["form_name", "form_generation"] + eda.STAT_COLUMNS])
display(forms.loc[forms["form_name"].isin(["arceus-normal", "arceus-fairy"]),
                  ["form_name", "pokemon_id", "introduction_games", "form_generation", "species_generation"]])

,form_name,form_generation,hp,attack,defense,special-attack,special-defense,speed
5,charizard,1,78,84,78,109,85,100
744,lycanroc-midday,7,75,115,65,55,65,112
924,maushold-family-of-four,9,74,75,70,65,75,111
1158,charizard-mega-x,6,78,130,111,130,85,100
1252,lycanroc-midnight,7,85,115,75,55,75,82
1335,lycanroc-dusk,7,75,117,65,55,65,110
1389,charizard-gmax,8,78,84,78,109,85,100
1450,maushold-family-of-three,9,74,75,70,65,75,111


,form_name,pokemon_id,introduction_games,form_generation,species_generation
492,arceus-normal,493,diamond-pearl,4,4
1109,arceus-fairy,493,x-y,6,4


**Constat :** les profils de Lougaroc diffèrent ; ceux de Famignol sont identiques. Dracaufeu et Dracaufeu Gigamax ont les mêmes six statistiques, mais leurs formes sont introduites respectivement en générations 1 et 8. Arceus Fée apparaît après Arceus Normal tout en partageant sa variété.

**Conclusion :** ni les statistiques identiques ni une même variété ne justifient de supprimer une forme. Les statistiques et les générations sont renseignées pour les 1 579 formes. Aucune date de forme ne précède ici celle de son espèce ; ce contrôle ne constitue toutefois pas une validation historique indépendante de toutes les dates.

**Limite :** les statistiques sont celles de l’instantané, pas nécessairement celles qu’avait le Pokémon à son introduction. Une moyenne par forme donne aussi davantage de poids aux espèces ayant beaucoup de formes. Le niveau et la pondération devront être précisés pour chaque analyse.

## 3. Un type absent dans une table est-il inconnu ?

Les types des variétés et les types explicitement renseignés pour les formes restent séparés. La recherche suivante repère toutes les variétés concernées à partir des données, sans sélectionner les noms Arceus ou Silvallié dans la règle.

In [4]:
form_types = eda.inspect_form_types(forms)
display(form_types)

,form_id,form_name,pokemon_id,variety_types,explicit_form_types
492,493,arceus-normal,493,"(Normal,)",NaN
772,773,silvally-normal,773,"(Normal,)",NaN
1065,10041,arceus-bug,493,"(Normal,)","(Bug,)"
1066,10042,arceus-dark,493,"(Normal,)","(Dark,)"
1067,10043,arceus-dragon,493,"(Normal,)","(Dragon,)"
1068,10044,arceus-electric,493,"(Normal,)","(Electric,)"
1069,10045,arceus-fighting,493,"(Normal,)","(Fighting,)"
1070,10046,arceus-fire,493,"(Normal,)","(Fire,)"
1071,10047,arceus-flying,493,"(Normal,)","(Flying,)"
1072,10048,arceus-ghost,493,"(Normal,)","(Ghost,)"


**Constat :** 35 formes ont un type explicitement renseigné, et 1 544 n’en ont pas dans cette table partielle. Toutes disposent néanmoins de types enregistrés au niveau de leur variété. La recherche générale retrouve Arceus et Silvallié.

**Explication :** les formes Normal n’ont pas de type répété dans la table des formes ; les autres formes peuvent y avoir un type différent de celui de la variété. `???` pour `arceus-unknown` est un libellé explicite, pas un `NaN`. Cette entrée historique inutilisée est conservée et signalée : notre inventaire ne se limite pas aux formes normalement accessibles en jeu. [Description d’Arceus](https://bulbapedia.bulbagarden.net/wiki/Arceus).

**Proposition non appliquée :** utiliser les types explicites de la forme lorsqu’ils existent, sinon ceux de la variété, en gardant la provenance. Nous ne remplaçons aucune valeur à ce stade.

## 4. Comprendre les autres valeurs manquantes de PokéAPI

Le premier tableau recense les absences dans toutes les tables CSV collectées. Les autres cherchent leurs raisons. Une ligne du tableau des traductions est une espèce dans une langue, pas une forme supplémentaire.

In [5]:
display(eda.missing_pokeapi(data))
missing_context = eda.missing_pokeapi_context(data)
display(missing_context["habitat"])
display(missing_context["varieties"])
display(missing_context["labels"])
display(missing_context["translations"])

,table,champ,lignes,manquantes,texte_vide,pourcentage_manquant
0,pokemon_species,evolves_from_species_id,1025,541,0,52.8
1,pokemon_species,habitat_id,1025,639,0,62.3
2,pokemon_species,conquest_order,1025,825,0,80.5
3,pokemon,base_experience,1351,49,0,3.6
4,pokemon,order,1351,139,0,10.3
5,stats,damage_class_id,9,4,0,44.4
6,stats,game_index,9,2,0,22.2
7,pokemon_species_names,genus,12300,1535,0,12.5
8,pokemon_forms,form_identifier,1579,961,0,60.9


,especes,habitats_absents
generation_id,,
1,151,0
2,100,0
3,135,0
4,107,107
5,156,156
6,72,72
7,88,88
8,96,96
9,120,120


,varietes,experiences_absentes,ordres_absents
is_default,,,
0,326,49,132
1,1025,0,7


,nombre
Noms courts de forme absents,961
Dont identifiants complets disponibles,961
Dont formes marquées par défaut,961


,lignes,noms_absents,descriptions_absentes
local_language_id,,,
1,1025,0,7
2,1025,0,1025
3,1025,0,0
4,1025,0,0
5,1025,0,0
6,1025,0,17
7,1025,0,127
8,1025,0,120
9,1025,0,0


| Constat dans l’instantané | Explication et proposition de traitement |
|---|---|
| 639 habitats absents | Toutes les espèces des générations IV à IX sont concernées, contre aucune des générations I à III. Ne pas utiliser ce champ pour comparer toutes les générations sans source complémentaire ; ne pas imputer un habitat majoritaire. |
| 49 expériences de base absentes | Elles concernent uniquement des variétés non définies par défaut. La cause exacte de chaque absence n’est pas établie. Garder les valeurs manquantes : ce champ n’est pas nécessaire aux six statistiques. |
| 139 valeurs `order` absentes | Il s’agit d’un ordre de présentation, pas de l’identifiant. Utiliser les identifiants pour les jointures. |
| 961 libellés courts de forme absents | Leurs identifiants complets sont tous disponibles ; toutes ces formes sont marquées par défaut. Afficher `identifier`, sans supprimer les formes ni inventer un suffixe `normal`. |
| Descriptions `genus` manquantes selon la langue | Elles sont distinctes des noms. Les lignes françaises (`local_language_id = 5`) sont complètes ; choisir explicitement une langue et ne pas inventer de traduction. |
| 541 liens `evolves_from_species_id` absents | Aucun prédécesseur n’est renseigné dans ce champ ; ce n’est pas une statistique manquante. Conserver l’absence, sans construire de catégorie évolutive dans notre périmètre. |
| 825 `conquest_order` absents | Cet ordre concerne Pokémon Conquest, pas l’ensemble du Pokédex. Ne pas l’utiliser comme identifiant ou ordre universel. |
| Métadonnées manquantes dans `stats` | Les PV et la vitesse n’ont pas de classe physique/spéciale ; précision et esquive n’ont pas d’index renseigné. Ce ne sont pas des absences parmi les six valeurs de base étudiées. |

**Conclusion :** aucune suppression générale des lignes incomplètes ne convient. Elle éliminerait notamment les espèces récentes sans habitat et de nombreuses formes dont seul le nom court manque. Nous conservons les observations et distinguons les absences non applicables, les informations disponibles ailleurs et les véritables lacunes. [Définition des champs PokéAPI](https://pokeapi.co/docs/v2).

## 5. Que manque-t-il dans Showdown, et que peut-il nous apporter ?

Showdown servira à **vérifier les informations communes après fusion** et à ajouter **Paradoxe, Ultra-Chimère et tier**. Les statuts légendaire/fabuleux sont déjà disponibles dans PokéAPI.

Nous distinguons un champ omis, une valeur explicitement nulle et un contenu vide. Ces nombres portent sur les 1 518 fiches Showdown ; ils ne sont pas encore des taux de couverture des 1 579 formes PokéAPI.

In [6]:
showdown = data["showdown_pokedex"]
display(eda.missing_showdown(showdown))

,champ,fiches,champ_absent,valeur_nulle,contenu_vide
0,num,1518,37,0,0
1,baseStats,1518,37,0,0
2,types,1518,37,0,0
3,heightm,1518,37,0,0
4,weightkg,1518,37,0,0
5,tags,1518,1358,0,0
6,tier,1518,102,0,0
7,gen,1518,1351,0,0
8,baseSpecies,1518,1121,0,0
9,forme,1518,1121,0,0


**Constat :** 37 fiches omettent les champs communs tels que le numéro, les statistiques ou les types. Le tier manque dans 102 fiches, les tags dans 1 358 et la génération explicite dans 1 351. Les champs étudiés sont omis, pas remplis par `null` ou une liste vide. Ces absences n’ont pas toutes la même signification : certaines fiches renvoient à une autre fiche.

Nous examinons ces références sans recopier leur contenu dans les données. Les colonnes `tier_base` et `tier_combat` ci-dessous sont des valeurs candidates à examiner, pas des tiers attribués.

In [7]:
references = eda.inspect_showdown_references(showdown)
display(references["summary"])
display(references["tier_conflicts"])

,nombre
Fiches avec un champ commun absent,37
Dont explicitement cosmétiques,37
Dont référence contenant les cinq champs,37
Fiches sans tier explicite,102
Dont tier disponible dans baseSpecies,102
Désaccords entre références de tier,2
Fiches sans champ tags,1358
Dont référence contenant des tags,94
Fiches étiquetées Paradox,16
Fiches étiquetées Ultra Beast,11


,cle,baseSpecies,battleOnly,tier_base,tier_combat
99,ogerponhearthflametera,Ogerpon,Ogerpon-Hearthflame,OU,Uber
100,ogerponcornerstonetera,Ogerpon,Ogerpon-Cornerstone,OU,UUBL


**Les 37 fiches cosmétiques** renvoient toutes à une fiche contenant les cinq champs communs. Il s’agit d’informations non répétées. Un héritage documenté sera envisageable pour ces fiches explicitement cosmétiques ; il ne doit pas être généralisé aux formes qui changent de type ou de statistiques.

**Les 102 tiers absents** ont tous une valeur dans la fiche `baseSpecies`, mais la copier systématiquement serait incorrect : pour deux transformations d’Ogerpon, cette référence porte `OU`, alors que la référence de combat porte `Uber` ou `UUBL`. Le [moteur Showdown](https://github.com/smogon/pokemon-showdown/blob/master/sim/dex-species.ts) prévoit des règles d’héritage et des cas particuliers. La règle applicable devra être documentée avant traitement ; le moteur courant n’est pas une version figée avec notre JSON. `Illegal` est une valeur de contexte, pas une absence ni un motif d’exclusion.

**Les tags absents ne signifient pas automatiquement « non Paradoxe » ou « non Ultra-Chimère ».** Parmi ces fiches, 94 renvoient à une fiche contenant des tags. Cela donne des pistes d’héritage, sans garantir l’exhaustivité des étiquettes de base. Les 16 fiches étiquetées `Paradox` et les 11 étiquetées `Ultra Beast` sont des comptages de mentions, pas une classification complète validée. Conserver les tags bruts et un statut indéterminé tant que la catégorie n’a pas été vérifiée.

**Limites :** le tier dépend du contexte compétitif et de la date, pas seulement des statistiques de base. La génération des formes est renseignée via PokéAPI. Les champs structurels `baseSpecies` ou `forme` ne sont pas nécessaires sur toutes les fiches de base. Aucun remplissage automatique n’est appliqué ici.

## 6. Les mêmes statistiques suffisent-elles pour fusionner ?

Nous cherchons les fiches Showdown ayant le même numéro d’espèce et les mêmes six valeurs. Cette recherche est seulement un diagnostic : elle ne valide ni l’identité de la forme, ni son type, ni sa génération. Les candidats restent séparés de notre inventaire.

In [8]:
candidates = eda.inspect_showdown_candidates(forms, showdown)
display(candidates["summary"])
display(candidates["candidates"].loc[
    candidates["candidates"]["form_name"].isin(["charizard", "charizard-gmax"]),
    ["form_name", "showdown_name"]])

,nombre
Formes sans candidat numérique,0
Formes avec un candidat numérique,1236
Formes avec plusieurs candidats numériques,343


,form_name,showdown_name
6,charizard,Charizard
7,charizard,Charizard-Gmax
2518,charizard-gmax,Charizard
2519,charizard-gmax,Charizard-Gmax


**Constat :** aucune forme n’est sans candidat numérique, 1 236 en ont un et 343 en ont plusieurs. Dracaufeu et Dracaufeu Gigamax illustrent pourquoi l’égalité des statistiques ne suffit pas à reconnaître une forme. Même un candidat unique n’est pas une identité validée ; une forme historique inutilisée peut partager un profil avec des fiches jouables.

**Conclusion pour la préparation :** établir une correspondance de forme documentée, conserver toutes les formes PokéAPI par une jointure à gauche, puis comparer séparément les valeurs des deux sources. Les formes non rapprochées ou ambiguës restent présentes. Une vérification des statistiques ne sera pas indépendante si l’on a choisi la correspondance uniquement pour obtenir des statistiques égales. L’accord entre deux sources peut aussi refléter une origine commune de leurs informations.

## Bilan : ce que nous retenons pour la suite

- **Population :** 1 579 formes, avec leurs identifiants de variété et d’espèce ; aucun regroupement, filtre par défaut ou retrait des Gigamax.
- **Informations disponibles :** les six statistiques et la génération sont renseignées pour chaque forme. La provenance des types reste à expliciter lors de la préparation.
- **Valeurs manquantes :** conserver les absences non utiles/non applicables ; examiner les héritages de fiches explicitement documentés ; ne pas déduire une catégorie négative d’un tag absent. Aucun traitement proposé n’est encore appliqué.
- **Fusion Showdown :** vérifier l’identité des correspondances avant d’ajouter les catégories et le tier. Mesurer ensuite la couverture sur les formes, en distinguant absence de correspondance et champ absent dans une fiche identifiée.
- **Limites :** exhaustivité relative aux sources, entrées historiques inutilisées, statistiques actuelles plutôt qu’historiques, répétition des profils entre formes et poids des espèces riches en formes. Les talents et conditions d’évolution sont hors périmètre.

Le choix de conserver le détail permet ensuite des analyses à des niveaux moins détaillés sans perte initiale d’information. Il faudra toutefois préciser le niveau et la pondération de chaque comparaison. L’absence de données manquantes sur une variable ne prouve pas l’exactitude de toutes ses valeurs. Nous n’avons pas encore produit de résultat scientifique sur l’évolution des statistiques entre générations.